# Recomendador de Municípios Brasileiros por Perfil Agropecuário

**Adaptação didática do projeto Cap08 da pós-graduação em Ciência de Dados da Data Science Academy (DSA).**

Este notebook demonstra, de ponta a ponta, o sistema de recomendação content-based construído sobre dados públicos do IBGE. Percorre os cinco estágios do pipeline (coleta, features, vetorização, similaridade, recomendação) e demonstra o funcionamento com casos concretos.

## Contexto

O projeto original da DSA ensina construção de sistema de recomendação a partir da representação de itens como vetores em espaço de alta dimensão e do cálculo de similaridade por cosseno, usando como corpus 4.800 filmes da TMDB. Esta adaptação transporta exatamente o mesmo aparato conceitual para o domínio agropecuário brasileiro, respondendo à pergunta prática: *"quais municípios brasileiros têm perfil agropecuário similar a um município de referência?"*

**Fonte de dados**: Pesquisa da Pecuária Municipal (PPM) 2024, IBGE, tabela SIDRA 3939, combinada com a divisão territorial brasileira via API Localidades v1.

**Utilidade prática**: consultoria agrícola (identificar municípios análogos para benchmarking), análise setorial (agrupar territórios por perfil produtivo), planejamento comercial (mapear mercados similares em outras regiões).

## Pré-requisitos

Este notebook assume que o pipeline de coleta e processamento já foi executado:

```powershell
.\tasks.ps1 download-all
.\tasks.ps1 build-features
.\tasks.ps1 vectorize
```

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rec_agro_br import features, similarity, vectorize
from rec_agro_br.recommender import MunicipioRecommender

%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 100)

## 2. Visão geral do dataset

O dataset processado contém uma linha por município brasileiro (5.571 no total, IBGE 2024) e 32 colunas: 12 de contexto territorial vindas da API Localidades, 8 numéricas com o efetivo de rebanho por atividade (vindas da PPM), 8 categóricas de perfil quantitativo (`perfil_bovinocultura`, `perfil_avicultura`, ...), a atividade dominante (`especializacao`), a contagem e lista de atividades (`n_atividades`, `atividades_presentes`) e o campo textual final `tags` que será vetorizado.

In [ ]:
df = features.load_features_dataset()
print(f"Municípios: {len(df):,}")
print(f"Colunas:    {df.shape[1]}")
print(f"UFs:        {df['sigla_uf'].nunique()}")
print(f"Regiões:    {df['nome_regiao'].nunique()}")
df.head(3)

### 2.1 Distribuição de especialização agropecuária

A coluna `especializacao` identifica, para cada município com produção pecuária, qual é a atividade em que ele ocupa o maior percentil nacional. Municípios sem produção recebem a categoria `sem_producao_pecuaria` (majoritariamente urbanos e periféricos).

In [ ]:
contagem = df["especializacao"].value_counts()
contagem_pct = 100 * contagem / len(df)

resumo = pd.DataFrame({"municipios": contagem, "pct": contagem_pct.round(1)})
resumo.index.name = "especializacao"
resumo

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cores = sns.color_palette("viridis", n_colors=len(contagem))
contagem.plot(kind="barh", ax=ax, color=cores)
ax.set_xlabel("Número de municípios")
ax.set_ylabel("Especialização agropecuária")
ax.set_title("Distribuição de especialização pecuária no Brasil (PPM 2024)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 2.2 Exemplo de tag construída

O coração do feature engineering é a concatenação de todas as features contextuais e categóricas de um município em uma única string, análoga ao campo `tags` do projeto DSA original. Esta string é o "documento" que será tokenizado e vetorizado.

In [ ]:
for _, row in df[df["sigla_uf"] == "MG"].sample(3, random_state=42).iterrows():
    print(f"[{row['sigla_uf']}] {row['nome_municipio']}")
    print(f"   Especializacao: {row['especializacao']}")
    print(f"   Tags: {row['tags']}")
    print()

## 3. Do texto ao vetor: CountVectorizer com stemming português

Aqui reside o núcleo pedagógico da matéria de Álgebra Linear Aplicada. Cada município é representado como um vetor em $\mathbb{R}^v$, onde $v$ é o tamanho do vocabulário do corpus. Cada dimensão do vetor corresponde a um token único (palavra) do vocabulário; o valor na dimensão $j$ do vetor do município $i$ é o número de vezes que o token $j$ aparece nas tags de $i$.

O tokenizer usado aplica o RSLPStemmer (versão brasileira do stemmer para português) apenas a tokens simples, preservando intactos os compostos com underscore (`sul_sudoeste_de_minas`, `especializado_em_bovinocultura`), que representam entidades semanticamente unitárias.

In [ ]:
vec = vectorize.load_vectorizer()
X = vectorize.load_matrix()

print(f"Documentos (municípios):    {X.shape[0]:,}")
print(f"Dimensão do vocabulário:    {X.shape[1]:,}")
print(f"Densidade da matriz:        {100 * X.nnz / (X.shape[0] * X.shape[1]):.2f}%")

### 3.1 Top 20 tokens mais frequentes

Um bom sanity check: os tokens mais frequentes devem ser aqueles que aparecem no maior número de tags. Como cada município menciona uma região (5 opções) e uma UF (27 opções), esperamos ver esses tokens dominando o topo da distribuição.

In [ ]:
freq = np.asarray(X.sum(axis=0)).ravel()
vocab_inv = {v: k for k, v in vec.vocabulary_.items()}
idx_top = np.argsort(freq)[::-1][:20]

top_tokens = pd.DataFrame({
    "token": [vocab_inv[i] for i in idx_top],
    "frequencia": [int(freq[i]) for i in idx_top],
})
top_tokens

## 4. Similaridade cosseno: intuição geométrica

A similaridade cosseno entre dois vetores $\mathbf{u}, \mathbf{v} \in \mathbb{R}^v$ é definida como:

$$\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \cdot \|\mathbf{v}\|_2}$$

Ela mede o ângulo entre os vetores, não a diferença de magnitude. Para vetores não-negativos como bag-of-words, o resultado está em $[0, 1]$: 1 significa vetores paralelos (mesmas tags), 0 significa ortogonais (nenhuma tag em comum).

Vamos exercitar o conceito com Cambuquira/MG e três candidatos: um município próximo (mesma mesorregião), um distante geograficamente mas com perfil similar, e um radicalmente diferente (urbano).

In [ ]:
def indice_por_nome_uf(nome: str, uf: str) -> int:
    mask = (df["nome_municipio"] == nome) & (df["sigla_uf"] == uf)
    return int(df.index[mask][0])

idx_cambu = indice_por_nome_uf("Cambuquira", "MG")
idx_sao_paulo = indice_por_nome_uf("São Paulo", "SP")

print(f"Cambuquira (idx={idx_cambu}):")
print(f"  tags: {df.loc[idx_cambu, 'tags']}")
print()
print(f"São Paulo (idx={idx_sao_paulo}):")
print(f"  tags: {df.loc[idx_sao_paulo, 'tags']}")

### 4.1 Cálculo manual passo a passo

Vamos calcular a similaridade cosseno entre Cambuquira e São Paulo usando a implementação manual didática do módulo `similarity`, que replica a fórmula matemática:

In [ ]:
u = X[idx_cambu].toarray().ravel()
v = X[idx_sao_paulo].toarray().ravel()

produto_interno = np.dot(u, v)
norma_u = np.linalg.norm(u)
norma_v = np.linalg.norm(v)
cos_manual = produto_interno / (norma_u * norma_v)

print(f"Produto interno u · v:   {produto_interno:.2f}")
print(f"||u||₂:                  {norma_u:.4f}")
print(f"||v||₂:                  {norma_v:.4f}")
print(f"cos(θ) = u·v/(||u||||v||): {cos_manual:.4f}")
print()

cos_lib = similarity.cosine_similarity_pair(u, v)
print(f"Confirmação via similarity.cosine_similarity_pair: {cos_lib:.4f}")

### 4.2 Comparação com implementação vetorizada

Para operações em lote (todos-contra-todos), delegamos ao `sklearn.metrics.pairwise.cosine_similarity`, que é vetorizado e usa BLAS internamente. Para nossa matriz $5571 \times 215$, todos-contra-todos rodam em milissegundos.

In [ ]:
%%time
S_completa = similarity.cosine_similarity_matrix(X)
print(f"Shape da matriz de similaridade: {S_completa.shape}")
print(f"Memória: {S_completa.nbytes / 1024**2:.1f} MB")

## 5. Sistema de recomendação

A classe `MunicipioRecommender` empacota todo o pipeline em uma API amigável. Carregamento inicial faz I/O uma vez; consultas subsequentes são O(n) sobre 5.571 municípios.

In [ ]:
rec = MunicipioRecommender.load()

### 5.1 Consulta por nome

Cambuquira/MG é o município natal do autor deste projeto. Um sistema que funcione deve identificar como mais similares outros municípios do Sul de Minas com perfil pecuário parecido — bovinocultura leiteira, avicultura, propriedades familiares.

In [ ]:
def imprimir_recomendacoes(query_desc, resultados):
    print(f"\nTop {len(resultados)} similares a {query_desc}:\n")
    for r in resultados:
        esp = r.especializacao.replace("especializado_em_", "")
        print(f"  {r.similaridade:.4f}  [{r.uf}] {r.nome:30s} ({r.mesorregiao})  {esp}")

resultados = rec.recommend_by_name("Cambuquira", uf="MG", k=5)
imprimir_recomendacoes("Cambuquira/MG", resultados)

### 5.2 Buscando análogos em outras UFs

Uma aplicação interessante para consultoria é encontrar municípios análogos **fora** da UF do consultado — útil para expansão comercial ou benchmarking interestadual. O parâmetro `excluir_mesmo_uf=True` remove todos os municípios da mesma UF do query.

In [ ]:
resultados = rec.recommend_by_name("Cambuquira", uf="MG", k=5, excluir_mesmo_uf=True)
imprimir_recomendacoes("Cambuquira/MG (excluindo MG)", resultados)

### 5.3 Consulta por código IBGE

Alternativa útil quando se trabalha com integração de sistemas ou tabelas de mapeamento.

In [ ]:
# Uberlândia/MG = 3170107
resultados = rec.recommend_by_code(3170107, k=5)
imprimir_recomendacoes("código IBGE 3170107 (Uberlândia/MG)", resultados)

### 5.4 Consulta por tags customizadas

Também é possível montar manualmente uma string de tags e pedir os municípios que mais se aproximam desse perfil hipotético. Útil para responder perguntas do tipo: *"quais municípios do Sudeste têm alta bovinocultura e alta avicultura simultaneamente?"*

In [ ]:
perfil_hipotetico = "sudeste alta_bovinocultura alta_avicultura"
resultados = rec.recommend_by_tags(perfil_hipotetico, k=5)
imprimir_recomendacoes(f'perfil hipotético "{perfil_hipotetico}"', resultados)

### 5.5 Explicando uma recomendação

Para responder à pergunta *"por que este município foi recomendado?"*, o método `explain` decompõe a comparação em tokens compartilhados e distintos, mostrando as três métricas de distância (cosseno, euclidiana, manhattan) lado a lado.

In [ ]:
top1 = rec.recommend_by_name("Cambuquira", uf="MG", k=1)[0]
exp = rec.explain(
    query_ref="Cambuquira",
    recomendado_ref=top1.nome,
    query_uf="MG",
    recomendado_uf=top1.uf,
)

print(f"Query:       {exp.query_nome} ({exp.query_uf})")
print(f"Recomendado: {exp.recomendado_nome} ({exp.recomendado_uf})")
print()
print(f"Similaridade cosseno:  {exp.similaridade_cosseno:.4f}")
print(f"Distância euclidiana:  {exp.distancia_euclidiana:.4f}")
print(f"Distância Manhattan:   {exp.distancia_manhattan:.4f}")
print()
print(f"Tokens em comum ({len(exp.tokens_em_comum)}):")
for t in exp.tokens_em_comum:
    print(f"  - {t}")

## 6. Visualização: heatmap de similaridade entre municípios de referência

Um heatmap para inspecionar visualmente a estrutura de similaridade entre um conjunto de municípios brasileiros conhecidos. Municípios da mesma região tendem a formar blocos com valores altos; municípios distantes geograficamente ou com perfis contrastantes ficam com similaridade baixa.

In [ ]:
municipios_referencia = [
    ("Cambuquira", "MG"),
    ("Uberlândia", "MG"),
    ("Guaxupé", "MG"),
    ("São Paulo", "SP"),
    ("Ribeirão Preto", "SP"),
    ("Sinop", "MT"),
    ("Sorriso", "MT"),
    ("Porto Alegre", "RS"),
    ("Passo Fundo", "RS"),
    ("Manaus", "AM"),
    ("Salvador", "BA"),
    ("Fortaleza", "CE"),
]

indices = []
labels = []
for nome, uf in municipios_referencia:
    mask = (df["nome_municipio"] == nome) & (df["sigla_uf"] == uf)
    if mask.any():
        indices.append(int(df.index[mask][0]))
        labels.append(f"{nome[:12]}/{uf}")

X_ref = X[indices]
S_ref = similarity.cosine_similarity_matrix(X_ref)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    S_ref,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    vmin=0,
    vmax=1,
    ax=ax,
    cbar_kws={"label": "Similaridade cosseno"},
)
ax.set_title("Similaridade cosseno entre municípios de referência (PPM 2024)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Reflexão

O sistema exercita o núcleo conceitual do módulo Cap08 da pós-graduação da DSA — representação vetorial, vetorização com bag-of-words, similaridade cosseno, recomendação content-based — e os aplica a um problema real de domínio agropastoril brasileiro. Cada peça do pipeline preserva 100% da lógica matemática do projeto original, adicionando as adaptações necessárias ao contexto brasileiro (RSLPStemmer para português; tokenizer seletivo para preservar identificadores geográficos compostos; validação de cobertura pós-download para proteger contra falhas silenciosas da API SIDRA).

Do ponto de vista científico, as recomendações produzidas são semanticamente coerentes: municípios da mesma mesorregião e com perfis agropecuários similares dominam as recomendações top-k, mas o sistema também identifica análogos em outras regiões — exatamente o comportamento útil para consultoria e análise setorial. As limitações são igualmente identificáveis: o sistema não sabe distinguir bovinocultura de corte de bovinocultura leiteira (a PPM 3939 não faz essa desagregação), não considera aspectos climáticos ou de solo, e não tem informação de escala relativa entre atividades (município com 1000 bovinos e 10 galináceos é tratado quase igual ao inverso). Extensões futuras poderão incorporar dados da PAM (produção agrícola municipal) e da PEVS (silvicultura) para enriquecer o vetor de features.

## Próximas etapas

- **Fase 1.F**: escrita da apostila didática (`docs/apostila/`) explicando cada conceito passo a passo, no estilo do repositório `estudos-observabilidade`.
- **Fase 1.G (opcional, nível mestrado)**: extensão de validação espacial usando Moran's I sobre a matriz de similaridade, para responder cientificamente à pergunta: *"municípios com perfil agropecuário similar tendem a ser geograficamente próximos, ou há clusters agropecuários dispersos pelo território?"*